In [6]:
# 单元格 1: 安装依赖
!pip install -U FlagEmbedding transformers datasets accelerate -q

import torch
print(f"CUDA 是否可用: {torch.cuda.is_available()}")
print(f"当前 GPU 设备: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '无'}")


CUDA 是否可用: True
当前 GPU 设备: NVIDIA A10


In [1]:
import json
import random
from datasets import Dataset

def prepare_4k_dataset(input_file="rerank_train_data_4K.jsonl", val_ratio=0.05, seed=42):
    raw_groups = []
    
    # 1. 读取原始嵌套格式的 JSONL
    with open(input_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                item = json.loads(line)
                if "query" in item and "docs" in item and len(item["docs"]) > 0:
                    raw_groups.append(item)
            except json.JSONDecodeError:
                continue

    # 2. 按 Query 级别随机打乱并切分训练集和验证集（防止同一个 Query 的文档同时出现在训练和验证中）
    random.seed(seed)
    random.shuffle(raw_groups)
    
    val_size = int(len(raw_groups) * val_ratio)
    val_groups = raw_groups[:val_size]
    train_groups = raw_groups[val_size:]

    # 3. 拍平（Flatten）为 Pointwise 回归所需的单行结构
    def flatten_groups(groups):
        flat_data = []
        for g in groups:
            q = g["query"]
            for doc in g["docs"]:
                flat_data.append({
                    "query": q,
                    "text": doc["text"],
                    "label": float(doc["label"])  # 必须转为 float，否则 MSELoss 会报错
                })
        return flat_data

    train_records = flatten_groups(train_groups)
    val_records = flatten_groups(val_groups)

    # 打乱训练集行顺序，提升梯度更新随机性
    random.shuffle(train_records)

    print(f"原始 Query 意图组总数: {len(raw_groups)}")
    print(f"训练集展开样本数: {len(train_records)} 条 (来自 {len(train_groups)} 个 Query)")
    print(f"验证集展开样本数: {len(val_records)} 条 (来自 {len(val_groups)} 个 Query)")

    return Dataset.from_list(train_records), Dataset.from_list(val_records), val_groups

# 执行数据转换
train_dataset, val_dataset, raw_val_groups = prepare_4k_dataset("rerank_train_data_4K.jsonl")


原始 Query 意图组总数: 4092
训练集展开样本数: 19440 条 (来自 3888 个 Query)
验证集展开样本数: 1020 条 (来自 204 个 Query)


In [3]:
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments
)

# 1. 加载模型与分词器（开启 num_labels=1 回归打分模式）
model_name = "BAAI/bge-reranker-v2-m3"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=1, 
    ignore_mismatched_sizes=True
)

# 2. 批量分词编码
def tokenize_function(examples):
    return tokenizer(
        examples["query"], 
        examples["text"], 
        padding="max_length", 
        truncation=True, 
        max_length=512
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["query", "text"])
tokenized_val = val_dataset.map(tokenize_function, batched=True, remove_columns=["query", "text"])

# 3. 单卡 A10 (24GB) 优化超参数
training_args = TrainingArguments(
    output_dir="./web_reranker_4k_checkpoints",
    learning_rate=2e-5,
    num_train_epochs=2,                  # 2万条训练对跑 2 个 Epoch 效果最佳（约 4750 Steps）
    per_device_train_batch_size=4,       # A10 跑 512 长度显存安全水位
    per_device_eval_batch_size=8,        # 验证集无需计算梯度，Batch 可以翻倍
    gradient_accumulation_steps=2,       # 等效总 BatchSize = 8
    fp16=True,                           # 开启 A10 混合精度加速
    weight_decay=0.01,
    warmup_steps=0.1,                    # 【已修复】新版 transformers 统一使用 warmup_steps 支持比例或步数
    logging_steps=100,
    eval_strategy="steps",               # 定期评估验证集 Loss
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,                  # 最多保留 2 个最新模型，节省磁盘
    load_best_model_at_end=True,         # 训练结束后自动加载验证集 Loss 最低的最优模型
    metric_for_best_model="eval_loss",
    report_to="none"
)

# 4. 启动训练
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
)

print("🚀 开始在 A10 GPU 上微调 4K 真实业务数据...")
trainer.train()

# 5. 保存最优模型
final_model_path = "./web_reranker_4k_final"
trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)
print(f"✅ 训练完成！最优模型已保存至 {final_model_path}")


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Map:   0%|          | 0/19440 [00:00<?, ? examples/s]

Map:   0%|          | 0/1020 [00:00<?, ? examples/s]

🚀 开始在 A10 GPU 上微调 4K 真实业务数据...


Step,Training Loss,Validation Loss
500,0.128463,0.033035
1000,0.039119,0.007218
1500,0.029076,0.005486
2000,0.030448,0.009241
2500,0.012948,0.004474
3000,0.014216,0.007270
3500,0.013481,0.011342
4000,0.011460,0.006857
4500,0.009925,0.004187
4860,0.007011,0.004177


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (7 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ 训练完成！最优模型已保存至 ./web_reranker_4k_final


In [4]:
import numpy as np

def evaluate_reranker(model_path, val_groups):
    eval_tokenizer = AutoTokenizer.from_pretrained(model_path)
    eval_model = AutoModelForSequenceClassification.from_pretrained(model_path).cuda().eval()

    all_ndcg = []
    all_mrr = []

    with torch.no_grad():
        for group in val_groups:
            query = group["query"]
            docs = [d["text"] for d in group["docs"]]
            true_labels = [float(d["label"]) for d in group["docs"]]

            pairs = [[query, doc] for doc in docs]
            inputs = eval_tokenizer(pairs, padding=True, truncation=True, max_length=512, return_tensors="pt").to("cuda")
            pred_scores = eval_model(**inputs).logits.view(-1).cpu().numpy()

            # 按预测分数降序排列
            rank_indices = np.argsort(pred_scores)[::-1]
            reranked_labels = np.array(true_labels)[rank_indices]

            # 计算 NDCG@5（标签平移 +1 避免 -1 出现负增益）
            dcg = sum(((2 ** (lbl + 1)) - 1) / np.log2(r + 2) for r, lbl in enumerate(reranked_labels[:5]))
            ideal_labels = sorted(true_labels, reverse=True)
            idcg = sum(((2 ** (lbl + 1)) - 1) / np.log2(r + 2) for r, lbl in enumerate(ideal_labels[:5]))
            all_ndcg.append(dcg / idcg if idcg > 0 else 0.0)

            # 计算 MRR@10（将 2分和 3分视为有效命中）
            mrr = 0.0
            for r, lbl in enumerate(reranked_labels[:10]):
                if lbl >= 2.0:
                    mrr = 1.0 / (r + 1)
                    break
            all_mrr.append(mrr)

    print("==========================================")
    print(f"📊 独立验证集 ({len(val_groups)} 组 Query) 最终评测结果：")
    print(f"平均 NDCG@5: {np.mean(all_ndcg):.4f}")
    print(f"平均 MRR@10: {np.mean(all_mrr):.4f}")
    print("==========================================")

# 运行评估
evaluate_reranker("./web_reranker_4k_final", raw_val_groups)


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

📊 独立验证集 (204 组 Query) 最终评测结果：
平均 NDCG@5: 1.0000
平均 MRR@10: 1.0000
